# 01 — RAG Explained: What It Is & Why LLMs Need It


## What is RAG?

An LLM's knowledge is frozen at training time — it's baked into the model's weights and
can't change afterward. That's like taking a **closed-book exam**: whatever you know, you
know, and nothing on the desk in front of you can fill a gap. It can't answer questions
about anything private, anything newer than its training cutoff, or anything simply never
in its training data. Ask anyway, and it either guesses (hallucination) or refuses.

**Retrieval-Augmented Generation (RAG)** turns that into an **open-book exam**. Instead of
relying only on what the model memorized, RAG hands it the right page from the book at the
moment it's asked a question. Concretely, that's two steps glued together:

1. **Retrieve** — search an external knowledge source (docs, a database, a wiki) for the
   handful of passages most relevant to the question.
2. **Augment & Generate** — paste those passages into the model's prompt as context, then
   let the model generate its answer _from that evidence_ instead of from memory alone.

That's it — no retraining, no fine-tuning. Update the knowledge source and the very next
query sees the change, which is what makes RAG the default way to connect an LLM to
information it was never trained on. The rest of this series is really just this one loop
— retrieve, augment, generate — explored one component at a time: how retrieval finds the
right passages, how chunking prepares text for it, how vector databases store it at scale,
and so on.


## Architecture

![Architecture diagram](https://mermaid.ink/img/Zmxvd2NoYXJ0IExSCiAgICBRW1VzZXIgcXVlc3Rpb25dIC0tPiBFW0VtYmVkIHF1ZXN0aW9uPGJyLz5iZ2UtbTNdCiAgICBFIC0tPiBTWyhJbi1tZW1vcnkgdmVjdG9yIHN0b3JlPGJyLz41IEFjbWUgUm9ib3RpY3MgZG9jcyldCiAgICBTIC0tPnx0b3AtayBjaHVua3N8IEFbQXVnbWVudCBwcm9tcHQ8YnIvPndpdGggcmV0cmlldmVkIGNvbnRleHRdCiAgICBBIC0tPiBMW2xsYW1hMy4yXQogICAgTCAtLT4gUltHcm91bmRlZCBhbnN3ZXJd?theme=dark)


## Setup

Runs fully on local Ollama models, so no API key is needed. Pull both models once:

```bash
ollama pull llama3.2
ollama pull bge-m3
```


In [ ]:
from pathlib import Path

DATA_DIR = Path("data")
print(f"Data directory: {DATA_DIR.resolve()}")

## `naive_llm` — ask the LLM about Acme Robotics with NO context at all

Acme Robotics is a fictional company invented for this repo, so the model has never seen it
during pretraining. This proves the point on camera: without retrieval, the model either
hallucinates a plausible-sounding but wrong answer, or admits it doesn't know. Compare this
output to the `tiny_rag` output below.


In [ ]:
from langchain_ollama import ChatOllama

# Deliberately about a fictional company - the base LLM can't know this,
# which is the point of the demo
QUESTION = "Who started Acme Robotics, and how long does their best robot's battery last?"


def ask_naive(question: str) -> str:
    # temperature=0 for reproducible output when we compare naive vs RAG answers
    llm = ChatOllama(model="llama3.2", temperature=0)
    response = llm.invoke(question)
    return response.content

In [ ]:
print("=" * 70)
print("NAIVE LLM (no retrieval, no context)")
print("=" * 70)
print(f"Q: {QUESTION}\n")
answer = ask_naive(QUESTION)
print(f"A: {answer}")
print("\n^ Notice this is either a refusal or a hallucination — the model")
print("  has never seen the fictional 'Acme Robotics' company before.")

## `tiny_rag` — the smallest possible working RAG pipeline

The loop is: RETRIEVE the most relevant chunks -> AUGMENT the prompt with them -> GENERATE
an answer grounded in that context. No vector DB server, no chunking library — just
LangChain's `InMemoryVectorStore` over 5 tiny markdown files, so the mechanics are visible
end to end.


In [ ]:
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_ollama import OllamaEmbeddings

# Instruct the model to stick to the retrieved context instead of falling back on its own knowledge
RAG_PROMPT = """Answer the question using ONLY the context below. \
If the context doesn't contain the answer, say you don't know.

Context:
{context}

Question: {question}

Answer:"""


def build_store() -> InMemoryVectorStore:
    """Embed every doc in data/ and load it into an in-memory vector store."""
    embeddings = OllamaEmbeddings(model="bge-m3")
    store = InMemoryVectorStore(embeddings)

    docs, metadatas = [], []
    # sorted() for a deterministic file order -- otherwise glob() order can vary by OS/filesystem,
    # which would make "chunk 2" mean a different doc on different runs
    for path in sorted(DATA_DIR.glob("*.md")):
        docs.append(path.read_text())
        # keep the filename around so we can show provenance for each retrieved chunk
        metadatas.append({"source": path.name})

    # add_texts embeds all 5 docs in a single batched API call rather than one call per doc --
    # cheap here since the corpus is tiny, but this is the same batching instinct you want at scale
    store.add_texts(docs, metadatas=metadatas)
    print(f"Indexed {len(docs)} documents from {DATA_DIR}")
    return store


def ask_rag(store: InMemoryVectorStore, question: str, k: int = 3) -> str:
    # RETRIEVE - embed the question and pull the k most similar chunks
    results = store.similarity_search_with_score(question, k=k)
    print(f"\nRetrieved {len(results)} chunks:")
    for doc, score in results:
        print(f"  score={score:.3f}  source={doc.metadata['source']}")

    # AUGMENT - stitch retrieved chunks into the prompt, separated so the model can tell them apart
    context = "\n\n---\n\n".join(doc.page_content for doc, _ in results)
    prompt = RAG_PROMPT.format(context=context, question=question)

    # GENERATE - same naive LLM, but now grounded in the retrieved context
    llm = ChatOllama(model="llama3.2", temperature=0)
    return llm.invoke(prompt).content

In [ ]:
print("=" * 70)
print("TINY RAG (retrieve -> augment -> generate)")
print("=" * 70)
store = build_store()
print(f"\nQ: {QUESTION}")
answer = ask_rag(store, QUESTION)
print(f"\nA: {answer}")

## Run it end-to-end

Run `naive_llm` and `tiny_rag` side by side on the same question, so the difference
retrieval makes is obvious in one screen.


In [ ]:
print("Question:", QUESTION)

print("\n" + "=" * 70)
print("WITHOUT RAG")
print("=" * 70)
print(ask_naive(QUESTION))

print("\n" + "=" * 70)
print("WITH RAG")
print("=" * 70)
# rebuilt here rather than reusing the `store` from cell 10 -- this cell is meant to be a
# standalone "run everything" demo, and re-embedding 5 tiny docs is cheap enough not to matter
store = build_store()
print(ask_rag(store, QUESTION))

## Key concepts

1. **Knowledge cutoff & hallucination** — the model can't know about anything invented
   after training, like our fictional "Acme Robotics." It will guess or refuse.
2. **The retrieve → augment → generate loop** — the core mechanical pattern behind every
   RAG system, no matter how sophisticated it gets later in this series.
3. **Context window as the real constraint** — you can't just paste your whole knowledge
   base into every prompt; retrieval picks the few chunks that matter.
4. **RAG vs. fine-tuning vs. long-context** — RAG updates knowledge without retraining and
   without paying to process irrelevant tokens on every call; each approach has a place.


## Try this yourself

1. Edit `data/acme_policies.md` to change the return window from 45 to 60 days, rerun the
   notebook, and confirm the RAG answer updates immediately with zero retraining.
2. Ask a question whose answer isn't in any of the 5 docs (e.g. "What color is the
   Junebug?") and see the RAG pipeline correctly say it doesn't know instead of guessing.
3. Change `k` in `ask_rag()` from 3 to 1 and see whether the answer degrades — this is your
   first taste of why chunk count/size matters (full topic in Episode 03).


## Common pitfalls

- **Forgetting `k` matters**: retrieving too few chunks starves the model of context;
  retrieving too many wastes tokens and can dilute relevant information.
- **Assuming retrieval always finds the right chunk**: it's a similarity search, not magic —
  garbage or off-topic source docs still produce garbage answers.
- **No source diversity check here**: this toy example doesn't dedupe or balance sources —
  that becomes important once your corpus grows past a handful of files.
